# SEARCH IMPROVE

In [5]:
import os
import sys
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.append(project_root)
from dotenv import load_dotenv
load_dotenv()

True

# OpenSearch test

In [13]:
from app.core.aws_clients import get_opensearch_client
from app.services.embeddings.bedrock_service import embed_text

In [8]:
os_client = get_opensearch_client()
INDEX_NAME = os.getenv('OPENSEARCH_NEW_INDEX')

In [11]:
# conteo de índices
res = os_client.count(index=INDEX_NAME)
print(res)

{'count': 466, '_shards': {'total': 5, 'successful': 5, 'skipped': 0, 'failed': 0}}


## Test de búsquedas

1. Ejemplo de match con descripciones

Para búsqueda textual. Usa análisis lingüístico (stemming, tokenización, stopwords).

In [ ]:

query = {
    "query": {
        "match": {
            "unified_description": "piscina"
        }
    }
}

res = os_client.search(index=INDEX_NAME, body=query)

for hit in res["hits"]["hits"]:
    print(f"Score: {hit['_score']}, Title: {hit['_source']['title']}, Descripción: {hit['_source']['unified_description']}")


Score: 5.0251484, Title: Casa Test 2, Descripción: Casa con patio y piscina Dormitorio Principal Habitación para descanso Baño Principal Servicios higiénicos Cocina Área de cocina
Score: 4.0042906, Title: Mini-departamento moderno en Residencial La Salle II, Breña, Descripción: Área confortable con acceso a instalaciones compartidas como piscina y gimnasio. Ideal para quienes buscan el lujo de la ciudad sin dejar de lado la tranquilidad. Dormitorio Principal Habitación para descanso Baño Principal Servicios higiénicos Sala Área social Cocina Área de cocina


2. Ejemplo de búsqueda con embeddings

In [16]:
text_example= 'Busco una casa con 3 habitaciones en los olivos, 2 baños , sala y comedor. En santiago de Surco'
text_embedding = embed_text(text_example)

In [17]:
query = {
  "size": 3,
  "query": {
    "knn": {
      "description_embedding": {
        "vector": text_embedding,
        "k": 3
      }
    }
  }
}

In [18]:
res = os_client.search(index=INDEX_NAME, body=query)

In [28]:
for hit in res["hits"]["hits"]:
    print(f"Score: {hit['_score']}, Title: {hit['_source']['title']}, Descripción: {hit['_source']['unified_description']}",
          f"Price: {hit['_source']['price']}, Geolocation: {hit['_source']['geolocation']}")

Score: 0.006636993, Title: Casa elegante en Urbanización Sol de Oro, Los Olivos, Descripción: Alquiler de una hermosa y moderna casa en Avenida Trébol, Urbanización Sol de Oro, Los Olivos. Este espacio ofrece cómodo ambiente y acceso a servicios en Lima Metropolitana. Baño Principal Servicios higiénicos Sala Área social Price: 2183.0, Geolocation: {'lat': -12.00115716048272, 'lon': -77.0670474749214}
Score: 0.0059472937, Title: Casa confortable en Urbanización Cooviecma, Santiago de Surco, Descripción: Alquila una moderna y acogedora casa en la prestigiosa urbanización Cooviecma. Ideal para vivir a menos de 10 minutos del centro de Santiago de Surco. Dormitorio Principal Habitación para descanso Dormitorio 2 Habitación para descanso Dormitorio 3 Habitación para descanso Baño Principal Servicios higiénicos Cocina Área de cocina Price: 2367.0, Geolocation: {'lat': -12.141296137272713, 'lon': -77.00641869408621}
Score: 0.0059274407, Title: Mini-departamento confortable en Urbanización El 

3. Ejemplo de búsqueda con keywords

Para exact match

In [ ]:
query = {
  "size": 5,
  "query": {
    "term": {
      "property_type": "mini-departamento"
    }
  }
}

In [40]:
res = os_client.search(index=INDEX_NAME, body=query)

In [41]:
for hit in res["hits"]["hits"]:
    print(f"Score: {hit['_score']}, Title: {hit['_source']['title']}, Descripción: {hit['_source']['unified_description']}",
          f"Price: {hit['_source']['price']}, Geolocation: {hit['_source']['geolocation']}")

Score: 1.8184278, Title: Mini-departamento lujoso en Emporio Comercial de Gamarra, Descripción: Venta de un moderno y elegante mini-departamento en el pasaje Hernando de Luque, Urbanización El Porvenir, La Victoria. Ubicación privilegiada, a pocos metros del centro comercial Emporio Comercial de Gamarra. Dormitorio Principal Habitación para descanso Baño Principal Servicios higiénicos Sala Área social Cocina Área de cocina Price: 165593.0, Geolocation: {'lat': -12.064716110801635, 'lon': -77.01643300334294}
Score: 1.8184278, Title: Moderno y confortable departamento en Lima, Descripción: Aprovecha este excelente departamento moderno, situado en un barrio tranquilo de Lima. Se ofrece una vista hermosa desde el balcón, y todo el equipo es reciente y de calidad. Dormitorio Principal Habitación para descanso Baño Principal Servicios higiénicos Sala Área social Cocina Área de cocina Price: 837.0, Geolocation: {'lat': -12.089322061121354, 'lon': -77.03509424461032}
Score: 1.8184278, Title: M

In [32]:
len(res['hits']['hits'])

10

4. Bool Queries

Usas must, should, must_not y filter para combinar condiciones:

- must: condiciones obligatorias (como AND).
- should: condiciones opcionales (como OR).
- filter: no afecta el score, pero filtra resultados.
- must_not: condiciones que deben no cumplirse (exclusión).

In [60]:
text_example= 'Departamento con balcón , elegante, moderno en un barrio tranquilo, histórico, zona céntrica'
embed_text = embed_text(text_example)

In [ ]:
query = {
    "size": 5,
    "query":{
        "bool":{
            "must": [
                #{"term": { "property_type": "mini-departamento"} },
                {"term": {"operation_type": "alquiler"}},
                {"geo_distance": {"distance": "5km", "geolocation": {"lat": -11.961409039974946, "lon":  -77.07631895655102}}}
            ],
            "filter": [
                { "range": { "price": { "lte": 2000 } } }
            ],
            "should": [
                {
                    "knn": {
                        "description_embedding": {
                            "vector": embed_text,
                            "k": 20
                         }
                    }
                }
            ]
        }
    }
}

res = os_client.search(index=INDEX_NAME, body=query)

In [84]:
for hit in res["hits"]["hits"]:
    print(f"Score: {hit['_score']}, Title: {hit['_source']['title']}, Descripción: {hit['_source']['unified_description']}",
          f"Price: {hit['_source']['price']}, Geolocation: {hit['_source']['geolocation']}")

Score: 1.7565168, Title: Departamento lujoso en Los Jazmines del Naranjal, Lima, Descripción: Departamento moderno con estilo y comodidad en el exclusivo barrio Los Olivos, Lima. Amenidades incluyen jardines comunales, seguridad 24/7 y acceso a transporte público. Dormitorio Principal Habitación para descanso Baño Principal Servicios higiénicos Sala Área social Cocina Área de cocina Price: 1884.0, Geolocation: {'lat': -11.97693779607747, 'lon': -77.0781167466375}
Score: 1.7224176, Title: Mini-departamento moderno en Los Olivos, Santa Cruz, Lima, Descripción: Alquila un espacioso y cómodo mini-departamento ubicado en el corazón del distrito de Los Olivos, Santa Cruz, Lima. El departamento cuenta con una cocina moderna, sala de estar, dormitorio y baño completos. Dormitorio Principal Habitación para descanso Baño Principal Servicios higiénicos Sala Área social Cocina Área de cocina Price: 1104.0, Geolocation: {'lat': -12.004506200742703, 'lon': -77.07196294049412}
Score: 1.7224176, Title

# LLM Test

## Ejemplo de chat

In [35]:
test_conversation = """
user: Hola!
assistant: Hola , qué tal, soy tu agente inmobiliario de confianza, podrías decirme en qué estas interesado hoy?.
user: Estoy buscando alquilar un departamento.
assistant: Excelente!, Podrías comentarme sobre el distrito o zona en el cual te encuentras interesado? , ¿Cuál es tu presupuesto aproximado?, ¿Tienes alguna preferencia en particular?
user: Ok, me interesan los distritos de Surco o Miraflores. Estoy dispuesto a pagar alrededor de 2500 soles mensuales. Preferiría que tenga al menos 2 dormitorios y 2 baños, con sala y comedor y que sea en una zona céntrica. También me gustaría que tenga al menos 70m2.
"""

## Definición de clase Pydantic

In [36]:
from pydantic import BaseModel, Field, field_validator
from typing import Optional, List
from enum import Enum

class PropertyType(str, Enum):
    DEPARTAMENTO = "departamento"
    MINI_DEPARTAMENTO = "mini-departamento"
    CASA = "casa"
    OFICINA = "oficina"
    LOCAL = "local"

class OperationType(str, Enum):
    ALQUILER = "alquiler"
    VENTA = "venta"

class PropertySearchParams(BaseModel):
    """Esquema para parámetros extraídos de la conversación"""

    # Criterios básicos
    property_types: Optional[List[PropertyType]] = Field(
        default=[], 
        description='Lista de tipos de propiedades deseados (ej: ["departamento", "casa", "mini-departamento", "local", "oficina"])'
    )
    operation_type: Optional[OperationType] = Field(None, description='Tipo de operación: Opciones: "alquiler" o "venta"')

    # Rango de precios
    min_price: Optional[float] = Field(None, description='Precio mínimo expresado en moneda local')
    max_price: Optional[float] = Field(None, description='Precio máximo expresado en moneda local')

    # Ubicación
    location: Optional[List[str]] = Field(None, description='Lista ubicaciones deseadas en cualquier nivel: (Ej.: ciudad, distrito, zona)')

    # Características físicas
    min_bedrooms: Optional[int] = Field(None, description='Cantidad mínima de dormitorios requeridos')
    max_bedrooms: Optional[int] = Field(None, description='Cantidad máxima de dormitorios aceptables')
    min_bathrooms: Optional[int] = Field(None, description='Cantidad mínima de baños requeridos')
    min_area: Optional[float] = Field(None, description='Área mínima de la propiedad en m2 (metros cuadrados)')
    max_area: Optional[float] = Field(None, description='Área máxima de la propiedad en m2 (metros cuadrados)')
    min_floor: Optional[int] = Field(None, description='Piso mínimo (ej: 2, excluye el primer piso)')
    max_floor: Optional[int] = Field(None, description='Piso máximo deseado')

    # Areas de la casa
    house_spaces: Optional[List[str]] = Field(
        default=[],
        description = 'Lista de ambientes del hogar deseados (ej: ["comedor", "sala", "patio", "cocina", etc...])'
    )

    # Amenidades
    amenities_wanted: Optional[List[str]] = Field(
        default=[],
        description='Lista de equipamientos o servicios internos deseados (ej: ["piscina","gym","parking", etc...])'
    )

    # Cercanías
    nearby_places: Optional[List[str]] = Field(
        default=[],
        description='Lugares o servicios externos cercanos deseados (ej: ["gym", "colegio", "parque", "centro comercial", "hospital", etc...])'
    )

    #Exclusiones
    amenities_exclude: Optional[List[str]] = Field(
        default=[],
        description='Lista de amenidades internas que el cliente desea evitar'
    )
    nearby_places_exclude: Optional[List[str]] = Field(
        default=[],
        description='Lugares o servicios cercanos que el cliente quiere evitar'
    )
    house_spaces_exclude: Optional[List[str]] = Field(
        default=[],
        description = 'Ambientes internos no deseados'
    )

    #Validators
    @field_validator('max_price')
    @classmethod
    def validate_price_range(cls, v, values):
        """Valida que los rangos sean válidos"""
        min_price = values.data.get('min_price')
        if min_price and v is not None and v < min_price:
            raise ValueError('max_price debe ser mayor que min_price')
        return v

## Extracción de entidades

In [37]:
from app.core.aws_clients import get_langchain_bedrock_client
from langchain_core.prompts import ChatPromptTemplate
import json

In [38]:
model_list = [
    "amazon.nova-micro-v1:0",
    "amazon.nova-lite-v1:0"
]

model_params = {
    'max_tokens': 250,
    'temperature' : 0.0,
    'top_p' : 1
}

In [39]:
# Función para describir la clase
def describe_class(target_class) -> str:
    """Construimos el contexto de la data a partir de la clase requerida"""
    data_context = ""
    for key,value in dict(target_class.model_fields.items()).items():
        data_info = key +  ": " + value.description
        data_context += data_info + "\n"
    return data_context.strip("\n")

def get_sample_element():
    """Retorna un ejemplo de un elemento PropertySearchParams"""
    data_dict = {
        'property_types': ['departamento', 'casa'],
        'operation_type': 'alquiler',
        'min_price': 2000.0,
        'max_price': 3000.0,
        'location': ['Miraflores, Lima, Perú', 'Buenos Aires, Capital Federal, Argentina'],
        'min_bedrooms': 2,
        'max_bedrooms': None,
        'min_area': 70,
        'max_area': 100,
        'min_floor': None,
        'max_floor': None,
        'house_spaces': ['comedor', 'sala'],
        'amenities_wanted': ['coworking', 'gimnasio'],
        'nearby_places': ['hospital', 'supermercado'],
        'amenities_exclude': [],
        'nearby_places_exclude': [],
        'house_spaces_exclude': []
    }

    return dict(PropertySearchParams(**data_dict))
    


In [83]:
get_sample_element()

{'property_types': [<PropertyType.DEPARTAMENTO: 'departamento'>,
  <PropertyType.CASA: 'casa'>],
 'operation_type': <OperationType.ALQUILER: 'alquiler'>,
 'min_price': 2000.0,
 'max_price': 3000.0,
 'location': ['Miraflores, Lima, Perú',
  'Buenos Aires, Capital Federal, Argentina'],
 'min_bedrooms': 2,
 'max_bedrooms': None,
 'min_bathrooms': None,
 'min_area': 70.0,
 'max_area': 100.0,
 'min_floor': None,
 'max_floor': None,
 'house_spaces': ['comedor', 'sala'],
 'amenities_wanted': ['coworking', 'gimnasio'],
 'nearby_places': ['hospital', 'supermercado'],
 'amenities_exclude': [],
 'nearby_places_exclude': [],
 'house_spaces_exclude': []}

In [41]:
print(describe_class(PropertySearchParams))

property_types: Lista de tipos de propiedades deseados (ej: ["departamento", "casa", "mini-departamento", "local", "oficina"])
operation_type: Tipo de operación: Opciones: "alquiler" o "venta"
min_price: Precio mínimo expresado en moneda local
max_price: Precio máximo expresado en moneda local
location: Lista ubicaciones deseadas en cualquier nivel: (Ej.: ciudad, distrito, zona)
min_bedrooms: Cantidad mínima de dormitorios requeridos
max_bedrooms: Cantidad máxima de dormitorios aceptables
min_bathrooms: Cantidad mínima de baños requeridos
min_area: Área mínima de la propiedad en m2 (metros cuadrados)
max_area: Área máxima de la propiedad en m2 (metros cuadrados)
min_floor: Piso mínimo (ej: 2, excluye el primer piso)
max_floor: Piso máximo deseado
house_spaces: Lista de ambientes del hogar deseados (ej: ["comedor", "sala", "patio", "cocina", etc...])
amenities_wanted: Lista de equipamientos o servicios internos deseados (ej: ["piscina","gym","parking", etc...])
nearby_places: Lugares o 

In [131]:
PROMPT_1 = """
        Eres un experto en extracción de criterios de búsqueda inmobiliaria.
        
        Primero lee el mensaje de cliente, identifica todas las posibles menciones de entidades y finalmente clasifícalas dentro del siguiente formato: 
        
        **TIPO DE PROPIEDAD (CRÍTICO):**
        Extrae exactamente el tipo mencionado y mapea a estos valores:
        - "departamento", "depa" → "departamento"
        - "mini departamento", "mini-departamento", "mini depa" → "mini-departamento"  
        - "casa" → "casa"
        - "oficina" → "oficina"
        - "local", "local comercial" → "local"
        Si el usuario dice "departamento", ponlo en property_types como ["departamento"]

        **TIPO DE OPERACIÓN:**
        - "compra, en venta, adquisición" → operation_type: compra
        - "alquilar, arremdar, rentar" → operation_type: alquiler


        **UBICACIONES:**
        - Normaliza nombres de distritos (ej: "los olivos" → location: "Los Olivos")

        **AMBIENTES DE LA CASA**
        - ej: 'cocina', 'comedor', 'sala', 'patio', etc

        **AMENIDADES DENTRO DE LA CASA**
        - ej: 'piscina', 'coworking', 'gym', 'parking', etc

        **CERCANÍAS**
        - ej: 'biblioteca', 'supermercado', 'iglesia', 'parques', etc

        **CARACTERÍSTICAS:**
        - "2 dormitorios" → min_bedrooms: 2, max_bedrooms: 2
        - "al menos 2 dormitorios" → min_bedrooms: 2
        - "máximo 3 dormitorios" → max_bedrooms: 3

        **RANGOS DE PRECIO:**
        - "hasta X" → max_price: X
        - "desde X" → min_price: X  
        - "entre X y Y" → min_price: X, max_price: Y
        - "alrededor de X" → min_price: X*0.8, max_price: X*1.2
        
        **EXCLUSIONES (muy importante):**
        - "no quiero", "que no sea", "excepto", "sin" → usar campos *_exclude
        - "no primer piso" → min_floor: 2
        - "no muy alto" → max_floor: 5 (estimado)  

        
        Sé muy preciso con las exclusiones y rangos. No inventemos si la información no es explícita.
        """

PROMPT_2 = """
        Eres un experto en extracción de criterios de búsqueda inmobiliaria.
        
        Primero lee el mensaje de cliente, identifica todas las posibles menciones de entidades y finalmente clasifícalas dentro del siguiente formato: 

        formato: {class_description}

        ❗No completes campos por inferencia.  
        ❗Si el dato no está mencionado literalmente o con sinónimos claros, ignoralo.


        **RANGOS DE PRECIO:**
        Tener en cuenta que para el precio, si el cliente brinda un presupuesto aproximado, asumir un rango de precio minimo y máximo. (Palabras clave: "aprox", "alrededor", "masomenos", "redondea")
        - ej: "alrededor de X" → min_price: X*0.8, max_price: X*1.2

        **AMENIDADES y CERCANÍAS:**
        - Las AMENITIES_WANTED son áreas adicionales DENTRO de la propiedad, como piscina, gym, etc.
        - Los NEARBY_PLACES son servicios o instalaciones FUERA de la propiedad como piscinas, gimnasio, hospitales, comisarías, etc.
        Si no se entiende la diferencia en el contexto, podemos asignar el valor a ambos campos.
"""

PROMPT_3 = """
        Eres un experto en extracción de criterios de búsqueda inmobiliaria.

        Tu tarea es analizar cuidadosamente el mensaje del cliente, identificar todas las menciones relevantes y clasificarlas en el siguiente formato JSON:

        formato: {class_description}

        📌 REGLAS:
        - No completes campos por inferencia o supuestos.
        - Si un dato no se menciona explícitamente o con sinónimos evidentes, deja ese campo como `null` o una lista vacía.
        - Si un valor aplica a múltiples campos (por ambigüedad), inclúyelo en ambos.

        📌 RANGOS DE PRECIO:
        - Si el cliente da un valor aproximado, infiere un rango razonable:
        → Ej: "alrededor de 2000" → min_price = 1600, max_price = 2400
        Palabras clave: "aproximado", "alrededor", "más o menos", "redondea", "cerca de"

        📌 AMBIENTES vs AMENIDADES vs CERCANÍAS:
        - `house_spaces`: Espacios internos como "sala", "cocina", "comedor", "patio", "terraza".
        - `amenities_wanted`: Servicios o equipamientos DENTRO de la propiedad, como "piscina", "ascensor", "parking", "gym".
        - `nearby_places`: Servicios o sitios EXTERNOS a la propiedad, como "colegios", "hospitales", "centros comerciales", "comisarías".
        - Si el contexto no es claro, puedes duplicar el término en los campos posibles.
"""
## Chain of Thought (COT) prompting
PROMPT_4 = rf"""
        Eres un asistente inteligente especializado en bienes raíces. Tu función es extraer datos sobre las necesidades de cliente.

        Dada una conversación, ejecuta los siguientes pasos:

        1. Resume las necesidades de cliente y arma un texto que explique con detalles lo que el cliente está buscando.
                - Se lo más conciso posible.
                - No inventemos ni asumamos información no mencionada explícitamente.
                - Si el cliente cambia de opinión con respecto a algo, usar la información más reciente.

        2. Identificar las entidades relevantes:
                - Hagamos una lista de posibles entidades relevantes: ubicación, tipo de propiedad(casa, departamento, etc), superficie, presupuesto, etc.
                - Si se mencionan aproximados (rangos no definidos), creemos un rango prudente. Ejm: 'Presupuesto de 2500 soles' -> precio mínimo:2000 y precio máximo:3000
        
        3. Realizar Named Entity Recognition (NER):
                - En base a las entidades identificadas, devuelve un diccionario categorizado por tipo.
                - Solo rellenar los campos identificados. 
                - Para temas de presupuesto tener en cuenta lo siguiente:
                        - 'Presupuesto máximo de 2000 soles' → max_price: 2000
                        - 'Presupuesto mínimo de 1000 soles' → min_price: 1000
                        - Inexactos, ambiguos , asumir un rango: 'Presupuesto aproximado de 2000': → min_price:1500, max_price:2500
                - Ejemplo: {str(json.dumps(get_sample_element())).replace('{', '{{').replace('}','}}')}
"""

In [133]:
class ConversationToQueryExtractor:
    def __init__(self, model_id=model_list[0], model_parameters=model_params):
        self.llm = get_langchain_bedrock_client(model_id = model_id, **model_parameters )

        #LLM con structured output
        self.structured_llm = self.llm.with_structured_output(
            schema=PropertySearchParams
        )

        self.prompt = ChatPromptTemplate.from_messages(
            [
                ('system', self._get_system_prompt()),
                ('human', "Conversación:\n{conversation}\n\nExtrae los criterios de búsqueda:")
            ]
        )
    
    def _get_system_prompt(self):
        #return PROMPT_2.format(class_description=describe_class(PropertySearchParams))
        return PROMPT_4
    
    def extract(self, conversation_text:str) -> PropertySearchParams:
        """Extraer parámetros estructurados de la conversación"""

        chain = self.prompt | self.structured_llm

        try:
            result = chain.invoke({"conversation": conversation_text})
            return result
        except Exception as e:
            print(f"Error en extracción: {e}")
            return PropertySearchParams()


In [134]:
extractor = ConversationToQueryExtractor(model_id = model_list[0])

In [138]:
params = extractor.extract(test_conversation)

In [139]:
print(test_conversation)


user: Hola!
assistant: Hola , qué tal, soy tu agente inmobiliario de confianza, podrías decirme en qué estas interesado hoy?.
user: Estoy buscando alquilar un departamento.
assistant: Excelente!, Podrías comentarme sobre el distrito o zona en el cual te encuentras interesado? , ¿Cuál es tu presupuesto aproximado?, ¿Tienes alguna preferencia en particular?
user: Ok, me interesan los distritos de Surco o Miraflores. Estoy dispuesto a pagar alrededor de 2500 soles mensuales. Preferiría que tenga al menos 2 dormitorios y 2 baños, con sala y comedor y que sea en una zona céntrica. También me gustaría que tenga al menos 70m2.



In [140]:
dict(params)

{'property_types': [<PropertyType.DEPARTAMENTO: 'departamento'>],
 'operation_type': <OperationType.ALQUILER: 'alquiler'>,
 'min_price': 2000.0,
 'max_price': 3000.0,
 'location': ['surco', 'miraflores'],
 'min_bedrooms': 2,
 'max_bedrooms': None,
 'min_bathrooms': 2,
 'min_area': 70.0,
 'max_area': None,
 'min_floor': 1,
 'max_floor': 10,
 'house_spaces': [],
 'amenities_wanted': ['sala', 'comedor'],
 'nearby_places': [],
 'amenities_exclude': [],
 'nearby_places_exclude': [],
 'house_spaces_exclude': []}

## Test con resumen

In [146]:
resumen_conversacion = """
    Busco comprar un departamento o un minidepa en los distritos de Surco o Miraflores. Como máximo puedo pagar 2000 soles.
    Deseo que tenga al menos 2 dormitorios y 2 baños, pero no más de 5 dormitorios, con ambientes de sala y comedor. Y que se encuentre ubicado en una zona céntrica.
    El área dedbe ser de al menos 70m2 y debe encontrarse como mínimo en el piso 5. El edificio debería tener ascensor.
    Me encantaría que esté cerca a parques o centros comerciales.
    Evitar zonas cercanas a comisarías o hospitales.
"""

In [147]:
extractor = ConversationToQueryExtractor(model_id = model_list[0], model_parameters=model_params)
params = extractor.extract(resumen_conversacion)

In [148]:
dict(params)

{'property_types': [<PropertyType.DEPARTAMENTO: 'departamento'>,
  <PropertyType.MINI_DEPARTAMENTO: 'mini-departamento'>],
 'operation_type': None,
 'min_price': 1500.0,
 'max_price': 2000.0,
 'location': ['Surco', 'Miraflores'],
 'min_bedrooms': 2,
 'max_bedrooms': 5,
 'min_bathrooms': 2,
 'min_area': 70.0,
 'max_area': None,
 'min_floor': 5,
 'max_floor': 10,
 'house_spaces': ['sala', 'comedor'],
 'amenities_wanted': ['parque', 'centro comercial'],
 'nearby_places': [],
 'amenities_exclude': ['comisión de Policía', 'hospital'],
 'nearby_places_exclude': ['comisaría', 'hospital'],
 'house_spaces_exclude': []}

# GEOLOCATION

In [6]:
import requests

In [ ]:
def geocode_address(address):
    url = "https://nominatim.openstreetmap.org/search"
    params = {
        "q": address,
        "format": "json",
        "limit": 1
    }
    headers = {
        "User-Agent": "my-chatbot-app/1.0 (housycorp@gmail.com)"  # Cambia por algo tuyo
    }

    response = requests.get(url, params=params, headers=headers)
    
    if response.status_code != 200:
        raise Exception(f"Error en la solicitud: {response.status_code}")
    
    data = response.json()
    
    if not data:
        return None  # No se encontró dirección

    lat = float(data[0]["lat"])
    lon = float(data[0]["lon"])
    display_name = str(data[0]["display_name"])
    return lat, lon, display_name

In [ ]:
geocode_address('Covida, Los Olivos')

(-11.9942939,
 -77.0751659,
 'Residencial Covida, 1069-1070, Jirón Caraz, Urbanización Villa los Ángeles, Los Olivos, Lima, Lima Metropolitana, Lima, 15301, Perú')

# Chat con GEOCODE tool

In [4]:
from langchain_core.tools import tool
from app.core.aws_clients import get_langchain_bedrock_client
from typing import Tuple
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage
import requests

In [32]:
# Creamos una tool con el tool decorator
@tool
def geocode_address(address: str) -> Tuple[float, float, str]:
    """Obtiene una dirección exacta en base a un texto. También devuelve coordenadas."""
    url = "https://nominatim.openstreetmap.org/search"
    params = {
        "q": address,
        "format": "json",
        "limit": 1
    }
    headers = {
        "User-Agent": "my-chatbot-app/1.0 (housycorp@gmail.com)"  # Cambia por algo tuyo
    }

    response = requests.get(url, params=params, headers=headers)
    
    if response.status_code != 200:
        raise Exception(f"Error en la solicitud: {response.status_code}")
    
    data = response.json()
    
    if not data:
        return None  # No se encontró dirección

    lat = float(data[0]["lat"])
    lon = float(data[0]["lon"])
    display_name = str(data[0]["display_name"])
    return lat, lon, display_name


In [28]:
geocode_address.invoke({'address': 'Surco'})

(-11.8678175, -76.4576592, 'Surco, Huarochirí, Lima, Perú')

In [20]:
geocode_address.args

{'address': {'title': 'Address', 'type': 'string'}}

In [8]:
model_list = [
    "amazon.nova-micro-v1:0",
    "amazon.nova-lite-v1:0"
]

model_params = {
    'max_tokens': 250,
    'temperature' : 0.0,
    'top_p' : 1
}

In [8]:
# creación del llm
llm = get_langchain_bedrock_client(model_id=model_list[0], **model_params)

In [ ]:
# asociando tools
llm_with_tools = llm.bind_tools([geocode_address])

In [24]:
# creacion de chat 
messages = [SystemMessage(content="Eres un agente inmobiliario que extraer información de las necesidades de cliente. " \
                                "Por ahora, tu función es extraer la ubicación deseada por el cliente, para lo cual, " \
                                "puedews hacer uso de la tool: 'geocode_address' y solicitar al cliente confirmar la locación."),
            HumanMessage(content="Estoy buscando algo en Miraflores")]

In [67]:
# Llamada al modelo
ai_message = llm_with_tools.invoke(messages)
print(dict(ai_message.content[0]))
dict(ai_message.content[1])

{'type': 'text', 'text': '<thinking> El cliente ha mencionado que está buscando algo en Miraflores, pero no ha especificado si se trata de un tipo de propiedad o si prefiere detalles adicionales como presupuesto o habitaciónes requeridas. Primero debo extraer la ubicación exacta de "Miraflores" utilizando la herramienta \'geocode_address\' y luego corroborar con el cliente para obtener más información precisa. </thinking>\n'}


{'type': 'tool_use',
 'name': 'geocode_address',
 'input': {'address': 'Miraflores'},
 'id': 'tooluse_09Uzbb55RWip-IapUBtFHw'}

In [ ]:
ai_message

AIMessage(content=[{'type': 'text', 'text': '<thinking> El cliente ha mencionado que está buscando algo en Miraflores, pero no ha especificado si se trata de un tipo de propiedad o si prefiere detalles adicionales como presupuesto o habitaciónes requeridas. Primero debo extraer la ubicación exacta de "Miraflores" utilizando la herramienta \'geocode_address\' y luego corroborar con el cliente para obtener más información precisa. </thinking>\n'}, {'type': 'tool_use', 'name': 'geocode_address', 'input': {'address': 'Miraflores'}, 'id': 'tooluse_09Uzbb55RWip-IapUBtFHw'}], additional_kwargs={}, response_metadata={'ResponseMetadata': {'RequestId': '15429568-1ec3-479c-a387-bb6a9c86af3d', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Wed, 06 Aug 2025 16:21:07 GMT', 'content-type': 'application/json', 'content-length': '706', 'connection': 'keep-alive', 'x-amzn-requestid': '15429568-1ec3-479c-a387-bb6a9c86af3d'}, 'RetryAttempts': 0}, 'stopReason': 'tool_use', 'metrics': {'latencyMs': [703]},

In [71]:
ai_message.tool_calls

[{'name': 'geocode_address',
  'args': {'address': 'Miraflores'},
  'id': 'tooluse_09Uzbb55RWip-IapUBtFHw',
  'type': 'tool_call'}]

## Llamada Full

In [6]:
from app.services.tools.geo_lookup import geocode_address as geocode_address_imported

In [9]:
# asociando tools
llm = get_langchain_bedrock_client(model_id=model_list[1], **model_params)
llm_with_tools = llm.bind_tools([geocode_address_imported])

In [13]:
# creacion de chat 
messages = [SystemMessage(content="Eres un agente inmobiliario que extraer información de las necesidades de cliente. " \
                                "Por ahora, tu función es extraer la ubicación deseada por el cliente, para lo cual, " \
                                "puedews hacer uso de la tool: 'geocode_address' y solicitar al cliente confirmar la ubicación."),
            HumanMessage(content="Estoy buscando algo cerca al Parque Kennedy")]

In [14]:
try:
    ai_message = llm_with_tools.invoke(messages)
    if ai_message.tool_calls:
        print('Log: Llamada a tool')
        messages.append(ai_message)

        for tool_call in ai_message.tool_calls:
            tool_name = tool_call['name']
            tool_args = tool_call['args']

            match tool_name:
                case 'geocode_address':
                    _, _, result = geocode_address_imported.invoke(input=tool_args)
                case _:
                    result = None
            
            messages.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id = tool_call['id']
                )
            )

            final_response = llm_with_tools.invoke(messages)

except Exception as e:
    print("Error: ", e)

Log: Llamada a tool


In [ ]:
final_response

NameError: name 'final_response' is not defined

# Other tests

In [2]:
from app.models.PropertyLead import PropertySearchParams
import json

In [3]:
print(PropertySearchParams.describe_class())

property_types: Lista de tipos de propiedades deseados (ej: ["departamento", "casa", "mini-departamento", "local", "oficina"])
operation_type: Tipo de operación: Opciones: "alquiler" o "venta"
min_price: Precio mínimo expresado en moneda local
max_price: Precio máximo expresado en moneda local
location: Lista ubicaciones deseadas en cualquier nivel: (Ej.: ciudad, distrito, zona)
min_bedrooms: Cantidad mínima de dormitorios requeridos
max_bedrooms: Cantidad máxima de dormitorios aceptables
min_bathrooms: Cantidad mínima de baños requeridos
min_area: Área mínima de la propiedad en m2 (metros cuadrados)
max_area: Área máxima de la propiedad en m2 (metros cuadrados)
min_floor: Piso mínimo (ej: 2, excluye el primer piso)
max_floor: Piso máximo deseado
house_spaces: Lista de ambientes del hogar deseados (ej: ["comedor", "sala", "patio", "cocina", etc...])
amenities_wanted: Lista de equipamientos o servicios internos deseados (ej: ["piscina","gym","parking", etc...])
nearby_places: Lugares o 

In [4]:
PropertySearchParams.get_sample_element()

{'property_types': [<PropertyType.DEPARTAMENTO: 'departamento'>,
  <PropertyType.CASA: 'casa'>],
 'operation_type': <OperationType.ALQUILER: 'alquiler'>,
 'min_price': 2000.0,
 'max_price': 3000.0,
 'location': ['Miraflores, Lima, Perú',
  'Buenos Aires, Capital Federal, Argentina'],
 'min_bedrooms': 2,
 'max_bedrooms': None,
 'min_bathrooms': None,
 'min_area': 70.0,
 'max_area': 100.0,
 'min_floor': None,
 'max_floor': None,
 'house_spaces': ['comedor', 'sala'],
 'amenities_wanted': ['coworking', 'gimnasio'],
 'nearby_places': ['hospital', 'supermercado'],
 'amenities_exclude': [],
 'nearby_places_exclude': [],
 'house_spaces_exclude': []}

## Class Conversation

In [21]:
from app.services.stages.stage_logic import Conversation

In [37]:
chat_history = [{'role': 'user',
    'content': [{"text": "Hello"}]},
    {'role': 'assistant',
    'content': [{"text": "Hola, en que puedo ayudarte?"}]},
    #{'role': 'user',
    #'content': [{"text": "Busco una casa en Puerto Maldonado con 3 habitaciones como mínimo, deseo alquilar, mi presupuesto es de 2000 soles"}]}
]

In [38]:
conversation = Conversation(message_history=  chat_history)

In [39]:
response = conversation.get_lead()

In [40]:
response

PropertySearchParams(property_types=None, operation_type=None, min_price=None, max_price=None, location=None, min_bedrooms=None, max_bedrooms=None, min_bathrooms=None, min_area=None, max_area=None, min_floor=None, max_floor=None, house_spaces=None, amenities_wanted=None, nearby_places=None, amenities_exclude=None, nearby_places_exclude=None, house_spaces_exclude=None)

In [41]:
data_dict = dict(json.loads(json.dumps(dict(response))))
data_dict

{'property_types': None,
 'operation_type': None,
 'min_price': None,
 'max_price': None,
 'location': None,
 'min_bedrooms': None,
 'max_bedrooms': None,
 'min_bathrooms': None,
 'min_area': None,
 'max_area': None,
 'min_floor': None,
 'max_floor': None,
 'house_spaces': None,
 'amenities_wanted': None,
 'nearby_places': None,
 'amenities_exclude': None,
 'nearby_places_exclude': None,
 'house_spaces_exclude': None}

In [42]:
conversation.verify_lead()

False

In [43]:
conversation.get_missing_params()

['property_types', 'operation_type', 'location']

## Class Conversation with history from DYNAMODB

In [2]:
from app.services.dynamodb_queries import get_latests_messages
from app.services.chatbot_engine import convert_to_conversation
from app.services.stages.stage_logic import Conversation

In [3]:
user_id= 'Feliz'
conv_id = 'Feliz'
key_id = f'USER#{user_id}#CONV#{conv_id}'

In [6]:
message_history = get_latests_messages(key_id, 20)
conversation = convert_to_conversation(message_history)

In [7]:
conversation

[{'role': 'user',
  'content': [{'text': 'Busco departamento de 2 dormitorios en Miraflores para alquilar'}]},
 {'role': 'assistant',
  'content': [{'text': '**Te recomendamos las siguientes propiedades** : (Lista de propiedades)'}]},
 {'role': 'user', 'content': [{'text': '¿Cómo está el clima hoy?'}]},
 {'role': 'assistant',
  'content': [{'text': 'Soy experto en bienes raíces y estoy aquí para ayudarte a encontrar tu próximo hogar u oficina 🏢. ¿Qué tipo de propiedad necesitas y en qué zona?'}]},
 {'role': 'user', 'content': [{'text': '¿Cómo está el clima hoy?'}]},
 {'role': 'assistant',
  'content': [{'text': '¡Hola! Soy tu asistente especializado en búsqueda de propiedades 🏠. Te ayudo a encontrar la casa, departamento u oficina perfecta para ti. ¿Qué tipo de propiedad estás buscando?'}]},
 {'role': 'user',
  'content': [{'text': 'Busco departamento de 1 dormitorios en Miraflores para alquilar'}]},
 {'role': 'assistant',
  'content': [{'text': '**Te recomendamos las siguientes propie

In [8]:
conversation_item = Conversation(conversation)

In [9]:
conversation_item.conversation

[HumanMessage(content='Busco departamento de 2 dormitorios en Miraflores para alquilar', additional_kwargs={}, response_metadata={}),
 AIMessage(content='**Te recomendamos las siguientes propiedades** : (Lista de propiedades)', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='¿Cómo está el clima hoy?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Soy experto en bienes raíces y estoy aquí para ayudarte a encontrar tu próximo hogar u oficina 🏢. ¿Qué tipo de propiedad necesitas y en qué zona?', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='¿Cómo está el clima hoy?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='¡Hola! Soy tu asistente especializado en búsqueda de propiedades 🏠. Te ayudo a encontrar la casa, departamento u oficina perfecta para ti. ¿Qué tipo de propiedad estás buscando?', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Busco departamento de 1 dormitorios en Miraflores para a

In [6]:
from app.services.stages.stage1_extract import handle

In [7]:
chat_history = [{'role': 'user',
    'content': [{"text": "Hello"}]},
    {'role': 'assistant',
    'content': [{"text": "Hola, en que puedo ayudarte?"}]},
    #{'role': 'user',
    #'content': [{"text": "Busco una casa en Puerto Maldonado con 3 habitaciones como mínimo, deseo alquilar, mi presupuesto es de 2000 soles"}]}
]

In [8]:
result = handle(chat_history)

In [9]:
list(result.keys())

['conversation',
 'base_prompt',
 'conversation_class',
 'lead',
 'lead_verification',
 'next_stage',
 'missing_info',
 'final_prompt',
 'model_response']

In [12]:
result['lead']

PropertySearchParams(property_types=[], operation_type=None, min_price=None, max_price=None, location=['Paris', 'Lima', 'Buenos Aires'], min_bedrooms=None, max_bedrooms=None, min_bathrooms=None, min_area=None, max_area=None, min_floor=None, max_floor=None, house_spaces=[], amenities_wanted=[], nearby_places=[], amenities_exclude=[], nearby_places_exclude=[], house_spaces_exclude=[])

In [10]:
result

{'conversation': [{'role': 'user', 'content': [{'text': 'Hello'}]},
  {'role': 'assistant',
   'content': [{'text': 'Hola, en que puedo ayudarte?'}]}],
 'base_prompt': 'Simula ser un asesor inmobiliario que guía al usuario con PREGUNTAS para entender qué tipo de propiedad desea el cliente. Sé breve pero cordial y amigable. (máx 50 palabras).❗NO RECOMENDEMOS NADA, solo hagamos preguntas.❗Actualmente los datos FALTANTES son: {datos_faltantes} <- Pregunta por estos ❗ Como contexto ten en cuenta los datos que podemos recolectar y su descripción:{data_info}',
 'conversation_class': <app.services.stages.stage_logic.Conversation at 0x12baf790a30>,
 'lead': PropertySearchParams(property_types=[], operation_type=<OperationType.ALQUILER: 'alquiler'>, min_price=None, max_price=None, location=None, min_bedrooms=None, max_bedrooms=None, min_bathrooms=None, min_area=None, max_area=None, min_floor=None, max_floor=None, house_spaces=[], amenities_wanted=[], nearby_places=[], amenities_exclude=[], near